<a href="https://colab.research.google.com/github/muhammeddanisht/langgraph-agents/blob/main/langgraph_v5_memory_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Imports

In [ ]:
# ════════════════════════════════════════════
# V5 FINAL — ALL MEMORY TYPES + TOOLS
# ════════════════════════════════════════════

# Install required packages
!pip install -U langchain-groq

# ── IMPORTS ──────────────────────────────────
from typing import Literal                                # for router return values
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.runnables import RunnableConfig       # carries user_id into nodes
from langchain_core.tools import tool                     # to create tools (from v4)
from langchain_groq import ChatGroq                       # our Groq brain
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode                   # runs tools automatically (from v4)
from langgraph.checkpoint.memory import MemorySaver       # short-term diary
from langgraph.store.memory import InMemoryStore          # long-term facts book
from langgraph.store.base import BaseStore                # type hint for store in nodes
import json                                               # read/write facts as dictionary
import os
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00


 Setup

In [ ]:
# ── API KEY + LLM ─────────────────────────────
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(model="llama-3.1-8b-instant")   # same brain as v4

Tools

In [ ]:
# ── TOOLS (same as v4) ───────────────────────
@tool
def calculator(a: float, b: float) -> float:
    """Multiply two numbers. Use when user asks to multiply or calculate product."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """Get weather for a city. Use when user asks about weather."""
    weather = {
        "kerala": "32°C sunny",
        "delhi": "41°C hot",
        "mumbai": "28°C humid"
    }
    return weather.get(city.lower(), "Weather not found for that city")

tools = [calculator, get_weather]   # list of all tools

# llm_with_tools = brain that KNOWS about tools
# when user asks to multiply → brain says "use calculator"
llm_with_tools = llm.bind_tools(tools)


 System Prompts



In [ ]:
# ── SYSTEM PROMPTS ───────────────────────────

# Main agent prompt — {memories} gets replaced with actual facts later
AGENT_PROMPT = """You are a helpful personal assistant with memory.
You remember important facts about the user.

Here is what you know about them:
{memories}

You also have tools:
- calculator: multiplies two numbers
- get_weather: gets weather for a city

Use tools when needed. Use your memory to personalize responses."""

# Extraction prompt — tells LLM what to extract from conversation
EXTRACT_PROMPT = """Read this conversation and extract ONLY important facts about the user.
Important facts = name, age, city, goals, job, skills, preferences.
Ignore: greetings, tool results, math questions, weather questions, small talk.
Return ONLY a JSON object. No explanation. No extra text.
Example: {{"name": "Danish", "goal": "AI job", "city": "Mattannur"}}
If nothing important found, return: {{}}"""

 call_model Node

In [ ]:
# ── NODE 1: call_model ───────────────────────
# Reads facts book → answers user → uses tools if needed
def call_model(
    state: MessagesState,    # all messages (short-term diary auto-managed here)
    config: RunnableConfig,  # carries user_id
    store: BaseStore         # facts book (auto-injected by LangGraph)
) -> dict:

    # Get who is talking
    user_id = config["configurable"].get("user_id", "default")

    # Open their folder in facts book
    namespace = ("user_facts", user_id)

    # Read all facts from folder
    memories = store.search(namespace)

    # Turn facts into readable text
    if memories:
        facts = memories[0].value
        # turn {"name":"Danish","goal":"AI job"} into "name: Danish | goal: AI job"
        memory_text = " | ".join([f"{k}: {v}" for k, v in facts.items()])
    else:
        memory_text = "No information about user yet."

    # Build system prompt with actual memories filled in
    system_prompt = AGENT_PROMPT.format(memories=memory_text)

    # Answer user — using llm_with_tools so agent can use calculator/weather
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm_with_tools.invoke(messages)

    return {"messages": [response]}

save_memory Node

In [ ]:
# ── NODE 2: save_memory ──────────────────────
# Extracts important facts → writes to facts book
def save_memory(
    state: MessagesState,
    config: RunnableConfig,
    store: BaseStore
) -> dict:

    # Get user_id + namespace
    user_id = config["configurable"].get("user_id", "default")
    namespace = ("user_facts", user_id)

    # Get last two messages (user message + agent reply)
    recent_messages = state["messages"][-2:]

    # Turn into readable text for LLM
    conversation_text = ""
    for msg in recent_messages:
        if isinstance(msg, HumanMessage):
            conversation_text += f"Human: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            conversation_text += f"Assistant: {msg.content}\n"

    # Ask LLM to extract important facts
    extract_response = llm.invoke([
        SystemMessage(content=EXTRACT_PROMPT),
        HumanMessage(content=conversation_text)
    ])

    # Parse LLM response as JSON
    try:
        extracted_facts = json.loads(extract_response.content)
    except:
        extracted_facts = {}   # if LLM returned invalid JSON → empty

    # Save if something important found
    if extracted_facts:
        # Read existing facts so we don't lose old ones
        existing = store.search(namespace)
        old_facts = existing[0].value if existing else {}

        # Merge old + new facts together
        # {**old_facts, **extracted_facts} = combine two dictionaries
        merged_facts = {**old_facts, **extracted_facts}

        # Write to facts book
        store.put(namespace, "profile", merged_facts)
        print(f"💾 Memory saved: {merged_facts}")
    else:
        print("💭 Nothing important to save.")

    return {}   # save_memory doesn't change messages

 Router + Tool Node

In [ ]:
# ── TOOL NODE ────────────────────────────────
# Same as v4 — runs the actual tool when agent calls it
tool_node = ToolNode(tools, handle_tool_errors=True)

# ── ROUTER ───────────────────────────────────
# After call_model runs → where to go next?
# If agent wants to use tool → go to "tools"
# If agent is done answering → go to "save_memory"
def route_after_model(state: MessagesState) -> Literal["tools", "save_memory"]:
    last_message = state["messages"][-1]   # get agent's last reply

    # hasattr checks if message HAS tool_calls field
    # tool_calls is filled when agent wants to use a tool
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"       # agent wants tool → run tool
    return "save_memory"     # agent done → save memory

Build Graph

In [ ]:
# ── BUILD GRAPH ──────────────────────────────
graph = StateGraph(MessagesState)

# Add all nodes
graph.add_node("call_model", call_model)     # node 1: answer user
graph.add_node("tools", tool_node)           # node 2: run tools
graph.add_node("save_memory", save_memory)   # node 3: save facts

# Wire the roads
graph.add_edge(START, "call_model")          # START → answer first

# After call_model → our router decides next road
graph.add_conditional_edges(
    "call_model",          # from this node
    route_after_model,     # this function decides road
    {
        "tools": "tools",               # if tools → run tool
        "save_memory": "save_memory"    # if done → save memory
    }
)

# After tools run → go back to call_model (agent sees tool result → replies)
graph.add_edge("tools", "call_model")

# After saving memory → conversation done
graph.add_edge("save_memory", END)

# ── COMPILE WITH BOTH MEMORY TYPES ───────────
memory = MemorySaver()    # short-term diary
store = InMemoryStore()   # long-term facts book

app = graph.compile(
    checkpointer=memory,  # attach diary
    store=store           # attach facts book
)

print("✅ V5 graph compiled successfully!")

✅ V5 graph compiled successfully!


Tests

In [ ]:
# ════════════════════════════════════════════
# TEST 1: Introduce yourself + use calculator
# Proves: tools work + facts get saved
# ════════════════════════════════════════════
print("\n" + "="*50)
print("TEST 1 — Intro + Calculator")
print("="*50)

config_1 = {"configurable": {"thread_id": "chat_1", "user_id": "danish"}}

r1 = app.invoke(
    {"messages": [HumanMessage(
        content="Hi! My name is Danish. I want AI job. Also multiply 12 times 8."
    )]},
    config=config_1
)
print("Agent:", r1["messages"][-1].content)

# ════════════════════════════════════════════
# TEST 2: New thread → ask name
# Proves: long-term memory survived new thread
# ════════════════════════════════════════════
print("\n" + "="*50)
print("TEST 2 — New thread, same user")
print("="*50)

config_2 = {"configurable": {"thread_id": "chat_2", "user_id": "danish"}}

r2 = app.invoke(
    {"messages": [HumanMessage(content="Do you know my name and my goal?")]},
    config=config_2
)
print("Agent:", r2["messages"][-1].content)

# ════════════════════════════════════════════
# TEST 3: Same thread as Test 1 → ask about calculator
# Proves: short-term memory works within same thread
# ════════════════════════════════════════════
print("\n" + "="*50)
print("TEST 3 — Same thread as Test 1")
print("="*50)

r3 = app.invoke(
    {"messages": [HumanMessage(content="What was the multiplication result you gave me?")]},
    config=config_1   # SAME config as Test 1 → same diary
)
print("Agent:", r3["messages"][-1].content)


TEST 1 — Intro + Calculator
💾 Memory saved: {'name': 'Danish', 'goal': 'AI job'}
Agent: Nice to meet you, Danish! I'd be happy to help you with your AI job aspirations. What specific aspects of AI are you interested in? Also, I've calculated the product of 12 and 8 for you. Would you like to know more about the AI job market or something else?

TEST 2 — New thread, same user
💾 Memory saved: {'name': 'Danish', 'goal': 'AI job'}
Agent: I remember that your name is Danish and your goal is to get a job in AI. How can I assist you in achieving your goal today?

TEST 3 — Same thread as Test 1
💭 Nothing important to save.
Agent: 96.0
